# 1. Project Introduction — Road Crack Detection with Deep Learning

**Goal:** a binary image classifier that decides whether a road/pavement image is **Cracked** (label 1) or **Non-Cracked** (label 0), and a reusable function `predict_road_crack(image_path)` that works on a brand-new image.

**Approach:** transfer learning with **MobileNetV2** (ImageNet weights) → train a small classification head → optionally fine-tune the last layers at a low learning rate → evaluate once on a held-out test set.

**Dataset:** SDNET (Structural Defects Network) attached to this Kaggle notebook. The notebook *discovers* the dataset folders itself; no path is hard-coded.

**Design decisions you should know about**
* **No data leakage:** SDNET tiles are cut from larger concrete panels. When filenames reveal the source panel (e.g. `7001-115.jpg` → panel `7001`), whole panels are kept in a single split.
* **Class imbalance:** SDNET has far more non-cracked than cracked tiles, so accuracy alone is misleading. We use class weights and report precision, recall, F1, ROC-AUC and — most importantly — **false negatives** (cracks the model misses).
* **The test set is used once**, at the end. The decision threshold and the choice between the two training stages are made on the **validation** set only.
* **Honest scope:** SDNET contains close-up, 256×256 photos of concrete surfaces. Wide photos of whole roads are a different kind of image, so results on them can be worse than the test numbers here.

**How to run:** turn on the GPU (Settings → Accelerator) and **Internet** (needed once to download the ImageNet weights), then *Run All*.

# 2. Import Libraries

In [ ]:
import os, re, json, random, time
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, roc_curve, roc_auc_score, precision_recall_curve)

# ----------------------------- configuration (edit here) -----------------------------
SEED = 42
IMG_SIZE = (224, 224)          # MobileNetV2 default input size
BATCH_SIZE = 32
INITIAL_EPOCHS = 10            # head training (max)
FINE_TUNE_EPOCHS = 10          # fine-tuning (max). Total never exceeds 20 epochs
FINE_TUNE_LAYERS = 30          # how many of the last MobileNetV2 layers to unfreeze
MAX_IMAGES_PER_CLASS = None    # e.g. 3000 for a quick trial run; None = use everything
CHECK_ALL_IMAGES = True        # verify every file can be decoded (takes ~1-3 min for 55k images)
TARGET_RECALL = 0.95           # validation recall we want when choosing the decision threshold
FORCE_RETRAIN = False          # False: reuse a previously saved model in /kaggle/working instead of retraining
ALLOW_RANDOM_INIT = False      # False: stop with a clear message if ImageNet weights cannot be downloaded
SEARCH_ROOTS = ["/kaggle/input"]   # where to look for the dataset (falls back to the current folder)

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_PATH = WORK_DIR / "road_crack_detection.keras"
HEAD_CKPT_PATH = WORK_DIR / "best_head_stage.keras"
FT_CKPT_PATH = WORK_DIR / "best_finetune_stage.keras"
HISTORY_PATH = WORK_DIR / "training_history.json"
CLASS_MAP_PATH = WORK_DIR / "class_mapping.json"
CONFIG_PATH = WORK_DIR / "preprocessing_config.json"

CLASS_NAMES = ["Non-Cracked", "Cracked"]       # index = label (Non-Cracked = 0, Cracked = 1)

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("Working directory:", WORK_DIR.resolve())

# 3. GPU and Environment Check

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for g in gpus:
        try:
            tf.config.experimental.set_memory_growth(g, True)   # avoid grabbing all GPU memory at once
        except RuntimeError as e:
            print("Could not set memory growth:", e)
    print(f"GPU available: {[g.name for g in gpus]}  -> training will be fast.")
else:
    print("No GPU found -> training on CPU will work but is slow.")
    print("Tip: Settings -> Accelerator -> GPU, or set MAX_IMAGES_PER_CLASS (section 2) for a quick trial.")
import sys, platform
print("Python", sys.version.split()[0], "|", platform.platform())

# 4. Dataset Path Discovery
We never assume the dataset path. The code walks the input folders and looks for directories called *Cracked* and *Non-Cracked* (case, spaces, `-` and `_` are ignored). If several folders match (for example one pair per surface type) they are pooled.

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}     # formats TensorFlow can decode natively

def norm_name(name):
    return re.sub(r"[\s_\-]+", "", name.lower())

NON_CRACKED_NAMES = {"noncracked", "uncracked", "nocrack", "noncrack", "nocracks"}

def find_class_dirs(roots):
    cracked, non_cracked = [], []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, _ in os.walk(root):
            for d in dirnames:
                n = norm_name(d)
                if n == "cracked":
                    cracked.append(Path(dirpath) / d)
                elif n in NON_CRACKED_NAMES:
                    non_cracked.append(Path(dirpath) / d)
    return sorted(cracked), sorted(non_cracked)

def show_tree(root, max_depth=2):
    root = Path(root)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        if depth > max_depth:
            dirnames[:] = []
            continue
        print("  " * depth + (Path(dirpath).name or str(root)) + f"/  ({len(filenames)} files)")

roots = [r for r in SEARCH_ROOTS if os.path.isdir(r)] or [os.getcwd()]
cracked_dirs, non_cracked_dirs = find_class_dirs(roots)

if not cracked_dirs or not non_cracked_dirs:
    print("Could not find both class folders. What the notebook can see:")
    for r in roots:
        show_tree(r)
    raise FileNotFoundError("No 'Cracked' / 'Non-Cracked' folders found. Attach the SDNET dataset "
                            "(Add Input) or edit SEARCH_ROOTS in section 2.")

all_class_dirs = cracked_dirs + non_cracked_dirs
DATASET_ROOT = Path(os.path.commonpath([str(p) for p in all_class_dirs]))
print("Dataset root directory:", DATASET_ROOT)
print("\nCracked folder(s):")
for p in cracked_dirs: print("   ", p)
print("Non-Cracked folder(s):")
for p in non_cracked_dirs: print("   ", p)

# 5. Dataset Inspection
Collect every image path, label and (when possible) its source-panel group, then print counts, sizes and formats.

In [ ]:
def list_images(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

records = []   # one dict per image
for label, dirs in ((1, cracked_dirs), (0, non_cracked_dirs)):
    for d in dirs:
        for p in list_images(d):
            records.append({"path": str(p), "label": label, "stem": p.stem, "parent": d.parent.name})

# Optional subsampling for a quick trial run (same fraction logic for both classes, reproducible).
if MAX_IMAGES_PER_CLASS:
    rng = random.Random(SEED)
    kept = []
    for lab in (0, 1):
        items = [r for r in records if r["label"] == lab]
        rng.shuffle(items)
        kept += items[:MAX_IMAGES_PER_CLASS]
    records = kept
    print(f"NOTE: using at most {MAX_IMAGES_PER_CLASS} images per class (trial mode).")

records.sort(key=lambda r: r["path"])      # fixed order -> reproducible splits
n_crack = sum(r["label"] == 1 for r in records)
n_non = sum(r["label"] == 0 for r in records)
print(f"Cracked images     : {n_crack}")
print(f"Non-Cracked images : {n_non}")
print(f"Total images       : {len(records)}")
if not records:
    raise RuntimeError("The class folders contain no .jpg/.jpeg/.png/.bmp images.")
if min(n_crack, n_non) == 0:
    raise RuntimeError("One of the classes has no images - cannot train a binary classifier.")
print(f"Imbalance ratio (non-cracked : cracked) = {n_non / n_crack:.1f} : 1")

# Dimensions / formats from a random sample (reading all headers would be slow and unnecessary)
sample = random.Random(SEED).sample(records, min(500, len(records)))
sizes, formats, modes, unreadable = Counter(), Counter(), Counter(), 0
for r in sample:
    try:
        with Image.open(r["path"]) as im:
            sizes[im.size] += 1; formats[im.format] += 1; modes[im.mode] += 1
    except Exception:
        unreadable += 1
print(f"\nImage sizes (width, height) in a random sample of {len(sample)}:", dict(sizes.most_common(5)))
print("Formats:", dict(formats), "| Colour modes:", dict(modes), "| unreadable in sample:", unreadable)
print("Example file names:", [Path(r["path"]).name for r in records[:3]], "...")

# 6. Exploratory Data Analysis

In [ ]:
# 6a. Find corrupted / unreadable images (full decode of every file) and drop them.
def is_valid_image(path):
    try:
        with Image.open(path) as im:
            im.convert("RGB").load()        # forces a full decode, catches truncated files
        return True
    except Exception:
        return False

bad_files = []
if CHECK_ALL_IMAGES:
    t0 = time.time()
    for i, r in enumerate(records):
        if not is_valid_image(r["path"]):
            bad_files.append(r["path"])
        if (i + 1) % 10000 == 0:
            print(f"  checked {i + 1}/{len(records)} files ...")
    print(f"Checked {len(records)} files in {time.time() - t0:.0f}s.")
else:
    print("CHECK_ALL_IMAGES=False: skipping the full corruption check (unreadable files are still skipped during training).")

if bad_files:
    print(f"Found {len(bad_files)} corrupted/unreadable image(s); removing them. First few:")
    for b in bad_files[:5]: print("   ", b)
    bad = set(bad_files)
    records = [r for r in records if r["path"] not in bad]
else:
    print("No corrupted or unreadable images found.")

n_crack = sum(r["label"] == 1 for r in records)
n_non = sum(r["label"] == 0 for r in records)
print(f"\nUsable images: {len(records)}  (Cracked {n_crack}, Non-Cracked {n_non})")

In [ ]:
# 6b. Sample images from both classes
def show_samples(label, n=8, title=""):
    items = [r for r in records if r["label"] == label]
    picks = random.Random(SEED + label).sample(items, min(n, len(items)))
    cols = 4
    rows = int(np.ceil(len(picks) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    for ax in np.array(axes).reshape(-1):
        ax.axis("off")
    for ax, r in zip(np.array(axes).reshape(-1), picks):
        with Image.open(r["path"]) as im:
            ax.imshow(im.convert("RGB"))
            ax.set_title(f"{im.size[0]}x{im.size[1]}", fontsize=9)
    fig.suptitle(title, fontsize=14)
    plt.tight_layout(); plt.show()

show_samples(1, title="Sample CRACKED images")
show_samples(0, title="Sample NON-CRACKED images")

In [ ]:
# 6c. Class distribution
counts = [n_non, n_crack]
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(CLASS_NAMES, counts, color=["#4c9f70", "#d9534f"])
for b, c in zip(bars, counts):
    ax.text(b.get_x() + b.get_width() / 2, c, f"{c}\n({100 * c / len(records):.1f}%)", ha="center", va="bottom")
ax.set_ylabel("Number of images"); ax.set_title("Class distribution (all usable images)")
ax.set_ylim(0, max(counts) * 1.2)
plt.show()

ratio = n_non / n_crack
print("What the EDA tells us:")
print(f"* The classes are imbalanced ({ratio:.1f}:1). A model that always says 'Non-Cracked' would already reach "
      f"{100 * n_non / len(records):.1f}% accuracy, so accuracy alone is a poor measure here.")
print("* We therefore (1) weight the loss by class, (2) judge the model by recall, precision, F1 and AUC, and "
      "(3) pay special attention to false negatives (missed cracks).")
print("* Cracks are thin, low-contrast structures: augmentation must stay mild (no heavy distortion/colour shifts).")

# 7. Data Preprocessing
Every image goes through the same steps, in training **and** at prediction time:

1. read the file → decode as **RGB** (3 channels, even if the file is grayscale);
2. resize to **224 × 224**;
3. `MobileNetV2.preprocess_input` — scales pixels from 0…255 to **−1…1**, exactly what the pretrained network expects;
4. label: **Cracked = 1, Non-Cracked = 0**.

The functions below are the single source of truth: the prediction function in section 18 reuses them.

In [ ]:
def load_rgb_resized(path):
    """File path -> float32 tensor (224, 224, 3), pixel range 0..255, RGB. Pure TF ops, so it works inside tf.data."""
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)   # channels=3 forces RGB
    img = tf.image.resize(img, IMG_SIZE)                                 # bilinear, returns float32
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img

def decode_with_label(path, label):
    return load_rgb_resized(path), tf.cast(label, tf.float32)

# quick self-test on one real image
_test = load_rgb_resized(records[0]["path"])
print("Preprocessing check -> shape:", _test.shape, "| dtype:", _test.dtype,
      "| raw range:", float(tf.reduce_min(_test)), "-", float(tf.reduce_max(_test)))
_pp = preprocess_input(tf.identity(_test)[None])
print("After preprocess_input -> range:", float(tf.reduce_min(_pp)), "to", float(tf.reduce_max(_pp)), "(expected within -1..1)")

# 8. Train / Validation / Test Split (70 / 15 / 15, stratified, leakage-safe)
**Why grouping matters:** SDNET tiles are crops of the same concrete panel. If tile `7001-12` is in training and `7001-13` is in the test set, the model can "recognise the panel" instead of learning cracks and the test score becomes too optimistic.

**What we do:** derive a *source group* from the file name (`<panel id>-<tile id>.jpg` → `<panel id>`) and use `StratifiedGroupKFold` with 20 folds (5 % each): 3 folds → test, 3 folds → validation, 14 folds → training. This keeps (almost) the class proportions while keeping every panel inside exactly one split. If the file names carry no such information, the code says so and falls back to a plain stratified split.

In [ ]:
def derive_group(r):
    m = re.match(r"^(\d+)[-_]", r["stem"])
    return f"{r['parent']}|{m.group(1)}" if m else None

groups = [derive_group(r) for r in records]
coverage = np.mean([g is not None for g in groups])
if coverage >= 0.9:
    groups = [g if g is not None else f"single_{i}" for i, g in enumerate(groups)]
    print(f"Source grouping FOUND from file names for {100 * coverage:.1f}% of images "
          f"-> {len(set(groups))} groups. Examples: {sorted(set(groups))[:4]}")
    GROUPING_USED = True
else:
    groups = [f"single_{i}" for i in range(len(records))]
    print(f"WARNING: only {100 * coverage:.1f}% of file names look like '<panel>-<tile>'. "
          "No reliable source information -> falling back to a plain stratified split "
          "(tiles from the same panel may land in different splits).")
    GROUPING_USED = False

paths = np.array([r["path"] for r in records])
labels = np.array([r["label"] for r in records], dtype=np.int32)
groups = np.array(groups)

N_FOLDS = 20
if len(set(groups)) < 3 * N_FOLDS:
    raise RuntimeError(f"Only {len(set(groups))} distinct groups - too few for a 70/15/15 grouped split.")

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_of = np.full(len(paths), -1)
for k, (_, idx) in enumerate(sgkf.split(paths, labels, groups)):
    fold_of[idx] = k
assert (fold_of >= 0).all()

split_of = np.where(fold_of < 3, "test", np.where(fold_of < 6, "val", "train"))
splits = {s: np.where(split_of == s)[0] for s in ("train", "val", "test")}

train_paths, train_labels = paths[splits["train"]], labels[splits["train"]]
val_paths, val_labels = paths[splits["val"]], labels[splits["val"]]
test_paths, test_labels = paths[splits["test"]], labels[splits["test"]]

# ---- leakage checks ----
assert not (set(train_paths) & set(val_paths)) and not (set(train_paths) & set(test_paths)) and not (set(val_paths) & set(test_paths))
if GROUPING_USED:
    g_tr, g_va, g_te = (set(groups[splits[s]]) for s in ("train", "val", "test"))
    assert not (g_tr & g_va) and not (g_tr & g_te) and not (g_va & g_te), "A source group appears in two splits!"
    print("Leakage check passed: no file and no source panel appears in more than one split.")

print(f"\n{'split':<6}{'images':>8}{'share':>8}{'cracked':>9}{'non-cracked':>13}{'% cracked':>11}")
for name, lab in (("train", train_labels), ("val", val_labels), ("test", test_labels)):
    print(f"{name:<6}{len(lab):>8}{100 * len(lab) / len(labels):>7.1f}%{int(lab.sum()):>9}{int((lab == 0).sum()):>13}{100 * lab.mean():>10.1f}%")
for name, lab in (("validation", val_labels), ("test", test_labels)):
    if lab.sum() < 30:
        print(f"WARNING: only {int(lab.sum())} cracked images in {name} - metrics will be noisy.")

# class distribution per split
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(3); w = 0.38
non_c = [int((l == 0).sum()) for l in (train_labels, val_labels, test_labels)]
cr_c = [int((l == 1).sum()) for l in (train_labels, val_labels, test_labels)]
ax.bar(x - w / 2, non_c, w, label="Non-Cracked", color="#4c9f70")
ax.bar(x + w / 2, cr_c, w, label="Cracked", color="#d9534f")
ax.set_xticks(x); ax.set_xticklabels(["Train", "Validation", "Test"]); ax.set_ylabel("Images")
ax.set_title("Class distribution per split"); ax.legend(); plt.show()

# 9. Data Augmentation (training data only)
Mild, crack-friendly augmentation: horizontal flip, rotation up to about ±11°, zoom ±10 %, contrast ±10 %. Validation and test data are only resized and normalised. Augmentation happens on the fly, so no augmented copies are stored and none can leak into other splits.

In [ ]:
augmenter = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),      # fraction of a full turn: 0.03 * 360 = about +/-10.8 degrees
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augmentation")

AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(file_paths, file_labels, training):
    ds = tf.data.Dataset.from_tensor_slices((file_paths, file_labels))
    if training:
        ds = ds.shuffle(len(file_paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_with_label, num_parallel_calls=AUTOTUNE)
    # Safety net: silently drop any file TensorFlow still cannot decode (already screened in section 6).
    ds = ds.ignore_errors() if hasattr(ds, "ignore_errors") else ds.apply(tf.data.experimental.ignore_errors())
    ds = ds.batch(BATCH_SIZE)
    if training:                                              # augmentation ONLY for training
        ds = ds.map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)   # same preprocessing everywhere
    return ds.prefetch(AUTOTUNE)

train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels, training=False)
test_ds = make_dataset(test_paths, test_labels, training=False)

# Class weights (computed from TRAINING labels only) so the rare 'Cracked' class counts more in the loss.
n_tr = len(train_labels)
class_weight = {0: n_tr / (2.0 * (train_labels == 0).sum()), 1: n_tr / (2.0 * (train_labels == 1).sum())}
print("Class weights:", {k: round(float(v), 3) for k, v in class_weight.items()})

# Visual check: the same training image, augmented several times
demo_raw = tf.stack([load_rgb_resized(str(train_paths[int(np.argmax(train_labels == 1))]))] * 6)
demo_aug = augmenter(demo_raw, training=True).numpy().clip(0, 255).astype("uint8")
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, im in zip(axes, demo_aug):
    ax.imshow(im); ax.axis("off")
fig.suptitle("One cracked training image after random augmentation (check that cracks remain visible)")
plt.show()

# 10. MobileNetV2 Model Building
**Architecture**
```
Input 224x224x3  ->  MobileNetV2 (ImageNet weights, no top, frozen at first)
                 ->  GlobalAveragePooling2D   (feature maps -> one 1280-value vector)
                 ->  Dropout(0.3)             (reduces overfitting)
                 ->  Dense(128, ReLU)         (learns crack-specific combinations of features)
                 ->  Dropout(0.3)
                 ->  Dense(1, sigmoid)        (probability that the image is CRACKED)
```
**Why MobileNetV2?** It was pretrained on ImageNet, so its early layers already detect edges, lines and textures, which are exactly the building blocks of cracks. It uses depthwise-separable convolutions, so it is small (~3.4 M parameters), fast to train on a free Kaggle GPU and quick at inference, yet accurate. Freezing it first lets the new head learn without destroying the pretrained features.

In [ ]:
def build_model():
    try:
        base = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
    except Exception as e:
        print("Could not load ImageNet weights:", e)
        print("Fix: in the Kaggle notebook Settings turn Internet ON (weights are downloaded once), then re-run.")
        if not ALLOW_RANDOM_INIT:
            raise RuntimeError("ImageNet weights unavailable. Enable Internet or set ALLOW_RANDOM_INIT=True "
                               "(not recommended: accuracy will be much worse).") from e
        print("WARNING: continuing with RANDOM weights because ALLOW_RANDOM_INIT=True.")
        base = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights=None)
    base.trainable = False                                   # stage 1: frozen

    inputs = keras.Input(shape=IMG_SIZE + (3,), name="image")
    x = base(inputs, training=False)                         # keep BatchNorm layers in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="crack_probability")(x)
    return keras.Model(inputs, outputs, name="road_crack_mobilenetv2"), base

def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.BinaryAccuracy(name="accuracy"),
                 keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall"),
                 keras.metrics.AUC(name="auc")],
    )

model, base_model = build_model()
model.summary()
print("Trainable parameters (head only):", sum(int(np.prod(v.shape)) for v in model.trainable_variables))

# 11. Model Compilation
Adam optimizer, binary cross-entropy loss, metrics: accuracy, precision, recall (+ AUC for monitoring). Stage 1 uses learning rate 1e-3 (only the new head is trained).

In [ ]:
compile_model(model, lr=1e-3)

def make_callbacks(ckpt_path):
    return [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(str(ckpt_path), monitor="val_loss", save_best_only=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7, verbose=1),
    ]

# Reuse a previous run instead of retraining (unless FORCE_RETRAIN=True).
skip_training = FINAL_MODEL_PATH.exists() and HISTORY_PATH.exists() and not FORCE_RETRAIN
print("Existing trained model found - training will be skipped (set FORCE_RETRAIN=True to retrain)."
      if skip_training else "No saved model (or FORCE_RETRAIN=True) -> the model will be trained now.")

# 12. Model Training (stage 1: train the new head)
The validation set monitors progress; `EarlyStopping` stops when validation loss stops improving and restores the best weights, `ModelCheckpoint` saves the best model, `ReduceLROnPlateau` lowers the learning rate when progress stalls. Training and validation images never overlap (section 8).

In [ ]:
history_head = None
if not skip_training:
    history_head = model.fit(
        train_ds, validation_data=val_ds, epochs=INITIAL_EPOCHS,
        class_weight=class_weight, callbacks=make_callbacks(HEAD_CKPT_PATH), verbose=1,
    )
    h = history_head.history
    best = int(np.argmin(h["val_loss"]))
    print(f"\nStage 1 finished after {len(h['loss'])} epoch(s). Best epoch: {best + 1}")
    print(f"  train acc {h['accuracy'][best]:.4f} | val acc {h['val_accuracy'][best]:.4f} | "
          f"train loss {h['loss'][best]:.4f} | val loss {h['val_loss'][best]:.4f}")
else:
    print("Skipped (using the saved model).")

# 13. Fine-Tuning (stage 2)
We unfreeze the last 30 layers of MobileNetV2 and train with a very low learning rate (1e-5) so the pretrained features are only nudged towards cracks. BatchNormalization layers stay frozen (standard practice for small learning rates and stable statistics). Total epochs over both stages never exceed 20.

Afterwards we compare the stage-1 best model and the stage-2 best model **on the validation set** and keep the better one. The test set is not touched.

In [ ]:
history_ft = None
if not skip_training:
    base_model.trainable = True
    for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
        layer.trainable = False
    for layer in base_model.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    print("Unfrozen layers:", sum(l.trainable for l in base_model.layers), "of", len(base_model.layers))

    compile_model(model, lr=1e-5)                      # must re-compile after changing trainable flags
    done = len(history_head.history["loss"])
    history_ft = model.fit(
        train_ds, validation_data=val_ds,
        initial_epoch=done, epochs=INITIAL_EPOCHS + FINE_TUNE_EPOCHS,
        class_weight=class_weight, callbacks=make_callbacks(FT_CKPT_PATH), verbose=1,
    )

    # ---- model selection on VALIDATION data only ----
    candidates = {"fine-tuned": model}
    if HEAD_CKPT_PATH.exists():
        candidates["head-only"] = keras.models.load_model(HEAD_CKPT_PATH)
    val_scores = {}
    for name, m in candidates.items():
        res = m.evaluate(val_ds, verbose=0, return_dict=True)
        val_scores[name] = res["loss"]
        print(f"  {name:<11} validation loss = {res['loss']:.4f} | accuracy = {res['accuracy']:.4f} | recall = {res['recall']:.4f}")
    chosen = min(val_scores, key=val_scores.get)
    model = candidates[chosen]
    print("Selected model:", chosen)

    # merge both histories into one dict, remembering where fine-tuning started
    history = {k: list(v) for k, v in history_head.history.items()}
    for k, v in history_ft.history.items():
        history.setdefault(k, []).extend(v)
    history["fine_tune_start_epoch"] = done
    history["selected_model"] = chosen
else:
    model = keras.models.load_model(FINAL_MODEL_PATH)
    history = json.loads(HISTORY_PATH.read_text())
    print("Loaded saved model and history from", WORK_DIR)

# 14. Training Visualization

In [ ]:
n_ep = len(history["loss"])
ft_start = history.get("fine_tune_start_epoch")
epochs_axis = range(1, n_ep + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, (tr, va, title) in zip(axes, [("accuracy", "val_accuracy", "Accuracy"), ("loss", "val_loss", "Loss")]):
    ax.plot(epochs_axis, history[tr], "o-", label=f"Training {title.lower()}")
    ax.plot(epochs_axis, history[va], "s-", label=f"Validation {title.lower()}")
    if ft_start:
        ax.axvline(ft_start + 0.5, color="gray", linestyle="--", label="fine-tuning starts")
    ax.set_xlabel("Epoch"); ax.set_ylabel(title); ax.set_title(f"{title} per epoch"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

best = int(np.argmin(history["val_loss"]))
print(f"Epoch with the lowest validation loss: {best + 1}")
print(f"  Training accuracy   : {history['accuracy'][best]:.4f}")
print(f"  Validation accuracy : {history['val_accuracy'][best]:.4f}")
print(f"  Training loss       : {history['loss'][best]:.4f}")
print(f"  Validation loss     : {history['val_loss'][best]:.4f}")
gap = history["accuracy"][best] - history["val_accuracy"][best]
print("A large positive train-validation accuracy gap would indicate overfitting; here the gap is "
      f"{gap:+.4f}. (Training metrics include augmentation and dropout, so they can look lower than validation early on.)")

# 15. Model Evaluation (unseen test set)
First we choose the **decision threshold on the validation set** (never on the test set): the highest-precision threshold that still reaches the target recall (`TARGET_RECALL`, 95 %). Missing a crack is worse than a false alarm, so we deliberately favour recall. Then the test set is scored once.

In [ ]:
def collect_predictions(m, ds):
    """Returns (true labels, predicted probabilities) in dataset order."""
    ys, ps = [], []
    for x, y in ds:
        ps.append(m(x, training=False).numpy().ravel())
        ys.append(y.numpy().ravel())
    return np.concatenate(ys).astype(int), np.concatenate(ps)

def choose_threshold(y_true, prob, target_recall):
    prec, rec, thr = precision_recall_curve(y_true, prob)
    prec, rec = prec[:-1], rec[:-1]                       # align with thr
    ok = np.where(rec >= target_recall)[0]
    if len(ok):
        i = ok[np.argmax(prec[ok])]                       # best precision among thresholds meeting the recall target
        how = f"highest precision with validation recall >= {target_recall:.0%}"
    else:
        f2 = 5 * prec * rec / (4 * prec + rec + 1e-9)     # F2 weights recall more than precision
        i = int(np.argmax(f2)); how = f"max F2 score (recall {target_recall:.0%} not reachable)"
    return float(thr[i]), how

y_val, p_val = collect_predictions(model, val_ds)
THRESHOLD, how = choose_threshold(y_val, p_val, TARGET_RECALL)
print(f"Chosen decision threshold = {THRESHOLD:.3f}  ({how}; picked on VALIDATION data)")
print(f"Validation recall at that threshold: {recall_score(y_val, p_val >= THRESHOLD):.3f} | "
      f"precision: {precision_score(y_val, p_val >= THRESHOLD, zero_division=0):.3f}")

y_test, p_test = collect_predictions(model, test_ds)
print(f"\nTest set size: {len(y_test)} images ({int(y_test.sum())} cracked)")

def metrics_at(thr):
    pred = (p_test >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    return {"threshold": thr, "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0), "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "false_negative_rate": fn / max(fn + tp, 1)}

m_tuned, m_default = metrics_at(THRESHOLD), metrics_at(0.5)
auc_score = roc_auc_score(y_test, p_test)
print(f"\n{'':<24}{'threshold=' + format(THRESHOLD, '.3f'):>18}{'threshold=0.500':>18}")
for k in ("accuracy", "precision", "recall", "f1", "false_negative_rate"):
    print(f"{k:<24}{m_tuned[k]:>18.4f}{m_default[k]:>18.4f}")
print(f"{'missed cracks (FN)':<24}{m_tuned['fn']:>18d}{m_default['fn']:>18d}")
print(f"{'false alarms (FP)':<24}{m_tuned['fp']:>18d}{m_default['fp']:>18d}")
print(f"\nROC-AUC (threshold independent): {auc_score:.4f}")

pred_final = (p_test >= THRESHOLD).astype(int)
print("\nClassification report (chosen threshold):")
print(classification_report(y_test, pred_final, target_names=CLASS_NAMES, digits=4, zero_division=0))

**What each metric tells us about crack detection**
* **Accuracy** – share of all images classified correctly. With imbalanced data it can look high even for a weak detector; do not rely on it alone.
* **Precision** – of the images flagged *Cracked*, how many really are cracked. Low precision = many false alarms = wasted inspections.
* **Recall (sensitivity)** – of all truly cracked images, how many we caught. **This is the key safety metric**: 1 − recall is the false-negative rate.
* **F1-score** – harmonic mean of precision and recall; one number that is low if either is low.
* **Confusion matrix** – the raw counts: *false negatives* (cracked predicted as non-cracked) are the dangerous errors; *false positives* are the cheaper ones.
* **ROC curve / AUC** – how well the model *ranks* cracked above non-cracked across all thresholds (1.0 = perfect, 0.5 = guessing). It is independent of the threshold we pick.

# 16. Confusion Matrix and ROC Curve

In [ ]:
cm = confusion_matrix(y_test, pred_final, labels=[0, 1])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels([f"Pred {c}" for c in CLASS_NAMES]); ax.set_yticklabels([f"True {c}" for c in CLASS_NAMES])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=16,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_title(f"Confusion matrix (threshold = {THRESHOLD:.3f})")
plt.colorbar(im, ax=ax, fraction=0.046)

ax = axes[1]
fpr, tpr, _ = roc_curve(y_test, p_test)
ax.plot(fpr, tpr, lw=2, label=f"ROC curve (AUC = {auc_score:.4f})")
ax.plot([0, 1], [0, 1], "k--", label="random guessing")
ax.scatter([m_tuned["fp"] / max(m_tuned["fp"] + m_tuned["tn"], 1)], [m_tuned["recall"]], c="red", zorder=5,
           label=f"chosen threshold ({THRESHOLD:.2f})")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate (recall)")
ax.set_title("ROC curve (test set)"); ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True negatives {tn} | False positives {fp} | False negatives {fn} | True positives {tp}")
print(f"=> {fn} of {fn + tp} cracked test images were MISSED ({100 * fn / max(fn + tp, 1):.1f}% false-negative rate).")

# Look at a few missed cracks - what do the failures look like?
if len(y_test) == len(test_paths):
    fn_idx = np.where((y_test == 1) & (pred_final == 0))[0]
    if len(fn_idx):
        show = fn_idx[np.argsort(p_test[fn_idx])][:8]                  # the most confidently missed first
        fig, axes = plt.subplots(2, 4, figsize=(14, 7))
        for ax in axes.ravel(): ax.axis("off")
        for ax, i in zip(axes.ravel(), show):
            with Image.open(test_paths[i]) as im_:
                ax.imshow(im_.convert("RGB"))
            ax.set_title(f"missed crack, p(crack)={p_test[i]:.2f}", fontsize=9)
        fig.suptitle("False negatives (cracks the model missed)"); plt.show()
else:
    print("(Skipping false-negative gallery: some test files were skipped as undecodable, order is not guaranteed.)")

# 17. Model Saving
Saves the model, the training history, the class mapping and the preprocessing configuration (including the chosen threshold). Files in `/kaggle/working` appear in the notebook **Output** panel after *Save Version → Save & Run All*, from where you can download them.

In [ ]:
model.save(FINAL_MODEL_PATH)

if not skip_training:
    HISTORY_PATH.write_text(json.dumps({k: ([float(x) for x in v] if isinstance(v, list) else v)
                                        for k, v in history.items()}, indent=2))

CLASS_MAP_PATH.write_text(json.dumps({"class_to_index": {"Non-Cracked": 0, "Cracked": 1},
                                      "index_to_class": {"0": "Non-Cracked", "1": "Cracked"},
                                      "positive_class": "Cracked"}, indent=2))

CONFIG_PATH.write_text(json.dumps({
    "image_size": list(IMG_SIZE),
    "color_mode": "RGB",
    "resize_method": "tf.image.resize (bilinear)",
    "preprocessing": "tensorflow.keras.applications.mobilenet_v2.preprocess_input (pixels scaled to [-1, 1])",
    "decision_threshold": THRESHOLD,
    "threshold_selection": f"validation set: highest precision with recall >= {TARGET_RECALL}",
    "test_metrics_at_threshold": {k: float(v) for k, v in m_tuned.items()},
    "test_roc_auc": float(auc_score),
    "split_seed": SEED,
    "source_grouping_used": bool(GROUPING_USED),
}, indent=2))

for p in (FINAL_MODEL_PATH, HISTORY_PATH, CLASS_MAP_PATH, CONFIG_PATH):
    print(f"saved: {p}  ({p.stat().st_size / 1024:.1f} KB)")

# remove the intermediate stage checkpoints so Output only contains the final files
for p in (HEAD_CKPT_PATH, FT_CKPT_PATH):
    if p.exists():
        p.unlink()
print("\nDownload: Save Version (top right) -> Save & Run All -> open the Output tab of the saved version.")

# 18. Prediction Function
`predict_road_crack(image_path)` loads any image, converts it to RGB, resizes to 224×224, applies the same `preprocess_input`, runs the model and shows the image with the result.

**Threshold:** an image is called *Cracked* when `P(cracked) >= threshold`. The default is the value chosen on the validation set (stored in `preprocessing_config.json`). Raise it for fewer false alarms, lower it to miss fewer cracks; to re-tune, call `choose_threshold(y_val, p_val, target_recall)` with a different target recall (validation data only). You can also override it per call: `predict_road_crack(path, threshold=0.3)`.

**Confidence** is the model's probability for the predicted class. It is a model score, not a guarantee, and may be over-confident on images unlike the training data.

In [ ]:
# Load the model/config from disk if this section runs in a fresh session.
if "model" not in globals():
    model = keras.models.load_model(FINAL_MODEL_PATH)
_cfg = json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.exists() else {}
DEFAULT_THRESHOLD = float(_cfg.get("decision_threshold", 0.5))
INFER_SIZE = tuple(_cfg.get("image_size", list(IMG_SIZE)))

def _prepare_for_model(path):
    """Same steps as training: RGB -> resize 224x224 -> MobileNetV2 preprocess_input. Returns (1,224,224,3)."""
    try:
        img = load_rgb_resized(str(path))                      # exactly the training code path
    except Exception:                                          # formats TF cannot decode (e.g. WEBP, TIFF)
        arr = np.array(Image.open(path).convert("RGB"), dtype=np.float32)
        img = tf.image.resize(arr, INFER_SIZE)
    return preprocess_input(tf.identity(img)[None, ...])

def predict_road_crack(image_path, threshold=None, show=True):
    """Predict Cracked / Non-Cracked for one image file. Returns a dict and (optionally) displays the result."""
    thr = DEFAULT_THRESHOLD if threshold is None else float(threshold)
    path = Path(image_path)
    if not path.exists() or not path.is_file():
        raise FileNotFoundError(f"Image not found: '{image_path}'. Check the path (files in /kaggle/working or "
                                f"/kaggle/input/<your-dataset>/...).")
    try:
        with Image.open(path) as im:
            original = im.convert("RGB")                       # for display; also validates that it is an image
    except Exception as e:
        raise ValueError(f"'{image_path}' is not a readable image file: {e}")

    prob_crack = float(model(_prepare_for_model(path), training=False).numpy().ravel()[0])
    is_crack = prob_crack >= thr
    label = "CRACKED" if is_crack else "NON-CRACKED"
    confidence = (prob_crack if is_crack else 1.0 - prob_crack) * 100

    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.2f}%")
    print(f"(P(cracked) = {prob_crack:.4f}, decision threshold = {thr:.3f})")
    if show:
        plt.figure(figsize=(5.5, 5.5))
        plt.imshow(original); plt.axis("off")
        plt.title(f"Prediction: {label}\nConfidence: {confidence:.2f}%", fontsize=14,
                  color="#c0392b" if is_crack else "#1e8449")
        plt.show()
    return {"prediction": label, "confidence_percent": round(confidence, 2),
            "probability_cracked": round(prob_crack, 4), "threshold": thr}

print("predict_road_crack() is ready. Default threshold:", round(DEFAULT_THRESHOLD, 3))

# 19. New Image Testing

### How to get your own image into a Kaggle notebook (no `google.colab` here)
`from google.colab import files` only works in Google Colab, so it is **not** used. In Kaggle use one of these:
1. **Upload as a private Dataset (recommended):** *+ Add Input → Upload* (or kaggle.com → Datasets → New Dataset), upload your photo(s), attach the dataset to this notebook, then copy the file path from the *Input* panel, e.g. `/kaggle/input/my-road-photos/crack1.jpg`.
2. **Download from a URL** (needs Internet ON): use `download_image(url, "/kaggle/working/test_road.jpg")` below.

Two kinds of tests are shown, and they mean different things:
* **Part A – sample test images:** taken from our *held-out test split*. They were never used for training or validation, but they come from the **same dataset** as the training data. They only demonstrate that the function works.
* **Part B – independent real-world images:** photos **you** provide from outside SDNET. This is the honest check of how the model behaves on new data. Note that SDNET contains close-up 256×256 concrete surface photos, so wide photos of whole roads may be out of domain; crop a close-up of the surface for a fair test.

In [ ]:
import urllib.request

def download_image(url, dest):
    """Optional helper (Internet must be ON in Kaggle settings). Returns the saved path."""
    dest = Path(dest)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=30) as r, open(dest, "wb") as f:
        f.write(r.read())
    print("Downloaded to", dest)
    return str(dest)

# ---------------- Part A: sample test images (held-out test split, same source as training data) ----------------
print("=" * 70)
print("PART A - SAMPLE test images from the held-out TEST split (NOT independent real-world images)")
print("=" * 70)
sample_cracked = str(test_paths[int(np.where(test_labels == 1)[0][0])])
sample_non_cracked = str(test_paths[int(np.where(test_labels == 0)[0][0])])

print("\nSample CRACKED image (true label: Cracked):", sample_cracked)
_ = predict_road_crack(sample_cracked)
print("\nSample NON-CRACKED image (true label: Non-Cracked):", sample_non_cracked)
_ = predict_road_crack(sample_non_cracked)

In [ ]:
# ---------------- Part B: independent real-world images (you provide them) ----------------
print("=" * 70)
print("PART B - INDEPENDENT images (not from SDNET, never seen by the model)")
print("=" * 70)

# Configurable input path - change these to your own files.
image_path = "/kaggle/working/test_road.jpg"                      # any single new image
INDEPENDENT_CRACKED_IMAGE = "/kaggle/working/test_road_cracked.jpg"       # a road photo that HAS a crack
INDEPENDENT_NON_CRACKED_IMAGE = "/kaggle/working/test_road_clean.jpg"     # a road photo WITHOUT cracks

def run_if_exists(path, expected=None):
    if not Path(path).exists():
        print(f"[skipped] '{path}' does not exist yet. Upload an image (see the instructions above) and edit the path.")
        return None
    if expected:
        print(f"(You said this image is: {expected})")
    return predict_road_crack(path)

print("\n--- single image test ---")
run_if_exists(image_path)
print("\n--- independent CRACKED road image ---")
run_if_exists(INDEPENDENT_CRACKED_IMAGE, expected="Cracked")
print("\n--- independent NON-CRACKED road image ---")
run_if_exists(INDEPENDENT_NON_CRACKED_IMAGE, expected="Non-Cracked")

# 20. Final Results and Conclusion

In [ ]:
print("=" * 60)
print("FINAL RESULTS (held-out TEST set, never used for training or model selection)")
print("=" * 60)
print(f"Test images            : {len(y_test)}  (cracked: {int(y_test.sum())}, non-cracked: {int((y_test == 0).sum())})")
print(f"Decision threshold     : {THRESHOLD:.3f}  (chosen on validation data)")
print(f"Accuracy               : {m_tuned['accuracy']:.4f}")
print(f"Precision              : {m_tuned['precision']:.4f}")
print(f"Recall                 : {m_tuned['recall']:.4f}")
print(f"F1-score               : {m_tuned['f1']:.4f}")
print(f"ROC-AUC                : {auc_score:.4f}")
print(f"Missed cracks (FN)     : {m_tuned['fn']}  of {m_tuned['fn'] + m_tuned['tp']}  "
      f"({100 * m_tuned['false_negative_rate']:.1f}% false-negative rate)")
print(f"False alarms (FP)      : {m_tuned['fp']}  of {m_tuned['fp'] + m_tuned['tn']} non-cracked images")
print("\nSaved files in", WORK_DIR, ":")
for p in sorted(WORK_DIR.glob("*")):
    if p.is_file() and p.suffix in (".keras", ".json"):
        print("  ", p.name)

**Conclusion and limitations**
* A MobileNetV2 transfer-learning classifier was trained in two stages (frozen head, then low-LR fine-tuning), with mild augmentation on the training data only, class weighting for the imbalance, and a validation-selected decision threshold. The numbers printed above come from a test set that played no role in training or tuning; they are the results of **this run**, and will differ slightly with another seed or another data split.
* **False negatives matter most.** The threshold was deliberately set for high validation recall; lowering it further trades more false alarms for fewer missed cracks. In a real inspection workflow, flagged images should be reviewed by a person.
* **Leakage control depends on the file names.** If the notebook reported that no source grouping was found, tiles from the same concrete panel may appear in different splits and the test scores will be optimistic.
* **Domain limits.** SDNET is concrete (decks, walls, pavements) photographed close-up. The model has not been validated on asphalt roads, wide-angle street photos, shadows, wet surfaces, painted markings or different cameras. Test on your own independent images (Part B) before trusting it elsewhere.
* **Possible improvements:** larger backbones (EfficientNet), k-fold cross-validation, probability calibration, Grad-CAM to see where the model looks, and a crack-segmentation model if you also need crack location or width.